# BioReservoir tutorial

A hands-on tour of the code as it stands for `v0.2`: what the model does, how to run and read a classification run, how to compare runs, how the data and the held-out beats are prepared, how to run optimisation studies, how to check behaviour against `v0.1`, and which scripts make the figures.

| Section | What you will do |
|---|---|
| 1. The model | Drive a small reservoir with one beat; look at its states, its kernel and its drive |
| 2. A classification run | Write a run config, resolve it and run it |
| 3. Reading a run | Per-fold, per-class and timing metrics |
| 4. Comparing runs | Another readout, another reservoir, other beats; the run registry |
| 5. Data | The dataset, the task pools and the preprocessing options |
| 6. Held-out beats | What the three draws are and how they will be used |
| 7. Optimisation | Studies, the command line, two small studies |
| 8. Checking behaviour | Regression tests, benchmarks, presets, and changing the model on purpose |
| 9. Figures | Which script makes which figure, and which figures depend on the coupling |

**Before you start**

- Environment: `uv sync --extra dev` (see the README), then pick `venv/` as the notebook kernel. VS Code offers to install `ipykernel` into it the first time; it is not a project dependency.
- Data: `python -m utils.preprocessing` builds `data/ecg/ecg_data.npz` and `data/ecg/binary_ecg_data.npz`.
- Size: the examples use 50 oscillators and 300 beats, so the whole notebook runs in about ten minutes on a laptop. The headline runs use 1,000 oscillators and 4,015 to 37,714 beats, and take hours on a server.
- Outputs: everything goes to `results/tutorial/` (gitignored; delete it when you are done). The notebook keeps its own run registry there, so the repository's `registry.jsonl` is not touched, and it never evaluates held-out draw 3.
- Cells marked *needs the archive* read the frozen `v0.1` results (`BIORESERVOIR_ARCHIVE_DIR` in `config.env`); they skip themselves when the archive is not there.

In [ ]:
import json
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numba
import numpy as np
import pandas as pd
import reservoirpy as rpy
import yaml

ROOT = Path.cwd().resolve()
while not ((ROOT / "pyproject.toml").is_file() and (ROOT / "bioreservoir").is_dir()):
    if ROOT == ROOT.parent:
        raise RuntimeError("Start this notebook from inside the bio-reservoir repository.")
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Everything the notebook writes goes to results/tutorial/ (gitignored).
OUT = ROOT / "results" / "tutorial"
RUNS = OUT / "runs"
REGISTRY = OUT / "registry.jsonl"  # the repository's registry.jsonl stays untouched
os.environ["BIORESERVOIR_RESULTS_DIR"] = str(OUT)
os.environ["BIORESERVOIR_LOGS_DIR"] = str(OUT / "logs")

# Small reservoirs run much faster on a few threads: waking every thread at each input
# step costs more than the work itself.
numba.set_num_threads(min(4, numba.config.NUMBA_NUM_THREADS))

# Show warnings only (the training code raises the root logger to DEBUG in every fold).
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.WARNING)
logging.basicConfig(level=logging.WARNING, handlers=[handler], force=True)
rpy.verbosity(0)


def cli(*args: str, show: int = 40) -> str:
    """Run ``python -m <module> ...`` (or ``python <script>.py ...``) with this kernel's
    Python; print the last ``show`` lines of its output and return the whole output."""
    command = [sys.executable, *([] if args[0].endswith(".py") else ["-m"]), *args]
    print("$ python " + " ".join(command[1:]))
    env = {**os.environ, "MPLBACKEND": "Agg", "PYTHONPATH": str(ROOT),
           "NUMBA_NUM_THREADS": str(numba.get_num_threads())}
    done = subprocess.run(command, cwd=ROOT, env=env, capture_output=True, text=True)
    output = done.stdout + (done.stderr if done.returncode else "")
    print("\n".join(output.splitlines()[-show:]))
    if done.returncode:
        raise RuntimeError(f"exit code {done.returncode}")
    return done.stdout


def fresh(path: Path) -> Path:
    """Delete a tutorial output folder so that a cell can be run again."""
    shutil.rmtree(path, ignore_errors=True)
    return path


print("outputs in", OUT.relative_to(ROOT), "| numba threads", numba.get_num_threads())

## 1. How BioReservoir works

**One oscillator.** Each reservoir node is a colony of *E. coli* carrying the quorum-sensing clock of Danino et al. (2010). Its state is four concentrations: the proteins LuxI (`I`) and AiiA (`A`), the signal molecule AHL inside the cells (`Hi`) and AHL in the medium around the colony (`He`). LuxI makes AHL, AiiA breaks it down, and AHL switches on the promoter of both genes after a delay τ. That delayed negative feedback makes the colony oscillate. The equations, as coded in `bioreservoir/dde.py` (time in input steps; τ = 10 steps):

$$
\begin{aligned}
\dot A &= C_A\,\kappa\,P\!\left(H_i(t-\tau)\right) - \frac{\gamma_A A}{1 + f\,(A + I)}\\
\dot I &= C_I\,\kappa\,P\!\left(H_i(t-\tau)\right) - \frac{\gamma_I I}{1 + f\,(A + I)}\\
\dot H_i &= \frac{b\,I}{1 + k I} - \frac{\gamma_H A H_i}{1 + g A} + D\left(H_e + c\,x(t) - H_i\right)\\
\dot H_e &= -\frac{d}{1 - d}\,D\,(H_e - H_i) - \mu H_e
\end{aligned}
\qquad
P(H) = \frac{\delta + \alpha H^2}{1 + k_1 H^2},\qquad \kappa = 1 - (d/d_0)^4
$$

The kinetic constants are Danino's (`bioreservoir.dde.DEFAULT_DDE_PARAMS`). The outside world reaches a colony only through the drive $x(t)$: it is multiplied by `cell_coupling` ($c$) and added to the AHL the cells see, so input and coupling both act as extra extracellular AHL.

**The drive.** For oscillator $j$ at input step $t$:

$$
x_j(t) = s_{\mathrm{dde}}\Big(\underbrace{\textstyle\sum_m W^{\mathrm{in}}_{jm}\,u_m(t) + b_j}_{\text{input}} \;+\; s_{\mathrm{rc}}\,\underbrace{\textstyle\sum_k K_{jk}\,\big(I_k + H_{e,k}\big)(t-1)}_{\text{coupling}}\Big)
$$

- $u(t)$ is one sample of the z-scored beat. `Win` connects it to a fraction `input_connectivity` of the oscillators, with weights ±`input_scaling`. The bias $b$ is off in the headline runs.
- The coupling uses the observed state of every oscillator at the end of the previous input step (`output_variables`: I and He in the headline runs, weighted by `couple_vars`, 1 each by default).
- $K$ is an N × N kernel. The oscillators sit at random positions in the unit square; a weight is a Gaussian of the distance, cut off at the interaction range (`interaction_diameters` times the mean nearest-neighbour distance); each weight gets a random sign (positive with probability `p_excite`); a fraction `sparsity` of the weights is kept; and $K$ is scaled to spectral radius 1.
- `rc_scaling` ($s_{\mathrm{rc}}$) and `dde_scaling` ($s_{\mathrm{dde}}$) set the gains; `noise_in` and `noise_rc` add Gaussian noise to the input and to the coupled states.
- This coupling is abstract: signed, applied once per input step, and carried mostly by LuxI. It is not AHL diffusing between colonies, which would move only AHL, in proportion to concentration differences, inside every solver substep.

**Integration.** The drive is held for one input step, which RK4 integrates in `rk4_substeps` substeps (64 in the headline runs). The delayed `Hi` is read at every substep (the default) or once per input step (`delay_scheme="frozen"`, the `v0.1` solver). A node starts from a random low-expression state, runs `warmup` steps without input when it is built, and every beat starts again from that warm-up state (`run(x, reset=True)`).

**From states to a class.** A beat of 187 samples gives a trajectory of 187 rows, one value per oscillator and observed variable. The features are an aggregation of it (the `mean` over the 187 steps in the headline runs; also `final`, `last_N:K` and `first_N:K`). A scikit-learn classifier, the readout, maps the features to one of the five classes N, S, V, F and Q. The readout is the only part that is fitted.

The cell below builds a 50-oscillator version of the headline reservoir with the `v0.1` code path, since its gains were tuned for it, and drives it with one beat.

In [ ]:
from bioreservoir.dde import DEFAULT_DDE_PARAMS
from optimization.reservoirs import build_reservoir
from utils import config as settings
from utils.preprocessing import load_ecg_data

SEED = settings.SEED  # 6337: the seed of every reported run
CLASS_NAMES = ["N", "S", "V", "F", "Q"]

# The v0.1 headline reservoir (P2-distance trial 416) with 50 oscillators instead of 1,000
# and 24 RK4 substeps instead of 64. The names are those of a run config.
RESERVOIR = {
    "units": 50, "warmup": 50, "rk4_substeps": 24,
    "input_scaling": 0.0277, "input_connectivity": 0.528, "input_bias": False,
    "bias_scaling": 0.1, "rc_scaling": 0.00704, "dde_scaling": 0.000644,
    "cell_coupling": 12.75, "noise_in": 0.0144, "noise_rc": 0.24,
    "output_variables": "IHe",
    "topology_type": "distance", "interaction_diameters": 8.2, "fade_alpha": 2.8e-05,
    "sparsity": 0.576, "signed": True, "p_excite": 0.835,
}
# The v0.1 code path (part of the original_v0_1 preset, section 8).
V0_1 = {"geometry": "published", "delay_scheme": "frozen", "normalize_after_masks": False}

print("kinetic constants:", DEFAULT_DDE_PARAMS)
X_train, Y_train, X_test, Y_test = load_ecg_data(rows=200, seed=SEED)
beat, label = X_train[0], int(np.argmax(Y_train[0]))  # one z-scored beat, shape (187, 1)
reservoir, _ = build_reservoir({**RESERVOIR, **V0_1}, seed=SEED)
states = reservoir.run(beat, reset=True)  # (187 steps, 50 oscillators x 2 variables)
lux_i, ahl_e = states[:, 0::2], states[:, 1::2]  # each oscillator's I and He sit side by side
features = states.mean(axis=0)  # the "mean" aggregation: what the readout sees for this beat
print("beat", beat.shape, "| states", states.shape, "| features", features.shape)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 6.5), sharex=True)
axes[0].plot(beat[:, 0], color="black")
axes[0].set(ylabel="input u(t)", title=f"One beat (class {CLASS_NAMES[label]}) and six oscillators")
axes[1].plot(lux_i[:, :6])
axes[1].set(ylabel="LuxI (I)")
axes[2].plot(ahl_e[:, :6])
axes[2].set(ylabel="external AHL (He)", xlabel="input step (one ECG sample)")
plt.show()

**The kernel.** `build_reservoir` gets $K$ from `distance_matrix(...)`; calling it directly with the same seed gives the same matrix, and `return_coords=True` also returns the positions. This is $K$ as the corrected code builds it (one position per oscillator, scaled last). The `v0.1` code built it over 2N points and scaled it before the signs and sparsity, which left a spectral radius of 0.43 for the headline topology (see the CHANGELOG).

In [ ]:
from bioreservoir import distance_matrix
from bioreservoir.topology import spectral_radius

kernel = {key: RESERVOIR[key] for key in
          ("interaction_diameters", "fade_alpha", "sparsity", "signed", "p_excite")}
K, positions = distance_matrix(50, 50, seed=SEED, return_coords=True, **kernel)
K = K.toarray()
distance = np.linalg.norm(positions[:, None] - positions[None], axis=-1)
kept = K != 0
print(f"spectral radius {spectral_radius(K):.2f}; {kept.sum()} of {K.size} weights kept, "
      f"{np.mean(K[kept] > 0):.0%} of them positive")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(distance[kept], K[kept], s=5, c=np.where(K[kept] > 0, "tab:red", "tab:blue"))
axes[0].set(xlabel="distance between two oscillators", ylabel="weight",
            title="Kernel weights (red positive, blue negative)")
image = axes[1].imshow(K, cmap="RdBu_r", vmin=-abs(K).max(), vmax=abs(K).max())
axes[1].set(title="K: row j holds the oscillators that drive j")
fig.colorbar(image, ax=axes[1])
plt.show()

**Is the model in the regime it was designed for?** `bioreservoir.diagnostics` answers this without any classification:

- `drive_and_state_ranges`: how large the drive $x$ gets (the model description bounds it by 5×10⁻³), and how low and high the states go;
- `drive_shares`: which share of the drive comes from the input and which from the coupling through each observed variable.

`python scripts/analysis/drive_bounds.py <config>` does the same for a run config. Here are four versions of the same 50-oscillator reservoir, over ten beats.

In [ ]:
from bioreservoir.diagnostics import drive_and_state_ranges, drive_shares

versions = {
    "v0.1 code path": {**RESERVOIR, **V0_1},
    "corrected code, v0.1 gains": RESERVOIR,
    "corrected code, rc_scaling / 10": {**RESERVOIR, "rc_scaling": RESERVOIR["rc_scaling"] / 10},
    "uncoupled (rc_scaling 0)": {**RESERVOIR, "rc_scaling": 0.0},
}
regimes = {}
for name, params in versions.items():
    node, _ = build_reservoir(params, seed=SEED)
    ranges = drive_and_state_ranges(node, X_train[:10])
    shares = drive_shares(node, X_train[:10])
    regimes[name] = {"max |x|": ranges["drive_abs_max"],
                     "min Hi": ranges["state_min"]["Hi"], "min He": ranges["state_min"]["He"],
                     "max I": ranges["state_max"]["I"],
                     **{f"share {key}": value for key, value in shares.items()}}
print(pd.DataFrame(regimes).T.to_string(float_format="{:.3g}".format))

How to read the table:

- **`v0.1` code path.** The drive is of the order of the stated bound, the AHL levels dip slightly below zero (as the CHANGELOG reports for the full-size reservoir), and nearly all of the drive is coupling through LuxI.
- **Corrected code with the `v0.1` gains.** One position per oscillator and a true spectral radius of 1 make the coupling several times stronger. With the same gains, LuxI reaches millions and the AHL levels fall far below zero. The gains were tuned for the old construction and need re-tuning before the corrected model is used.
- **A ten times smaller `rc_scaling`** brings the corrected model back within the bound, with non-negative AHL.
- **Without coupling** the drive is the input alone, two orders of magnitude smaller.

Run these diagnostics whenever the coupling changes: classification scores alone do not show whether the model is in a physical regime.

**Other inputs, such as slow sensing.** `run` accepts any input sequence of shape (T, 1); nothing in the reservoir is specific to ECG. For slow inducer sensing, where the colonies act as a sensor with memory, the input is a slow inducer pattern over hours (one input step is one model time unit, about a minute). The feature is one end-point snapshot of the colonies (`aggregation: final`), read by a linear readout and compared with a memoryless sensor that only sees the inducer at the end.

This repository has only the ECG task so far. A slow-sensing task needs an episode generator and its own task type in the run framework, which today loads ECG pools in `load_task` (`training/run.py`). Note also that the input weights are signed, so a non-negative inducer still pushes some colonies down; a model of an inducer would use non-negative input weights.

In [ ]:
steps = np.arange(240)  # four hours at about one model minute per step
inducer = (1 + np.sin(2 * np.pi * steps / 120))[:, None]  # a slow, non-negative pattern
response = reservoir.run(inducer, reset=True)
end_point = response[-1]  # one snapshot of every oscillator at the end: the feature vector

fig, axes = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
axes[0].plot(inducer[:, 0], color="black")
axes[0].set(ylabel="inducer")
axes[1].plot(response[:, 0:12:2])
axes[1].set(ylabel="LuxI (I)", xlabel="input step")
plt.show()
print("end-point features:", end_point.shape)

## 2. A classification run

The run framework (`training/run.py`) turns one YAML config into one run folder. A config has these sections; anything left out takes the default that `resolve` shows.

| Section | Keys | Notes |
|---|---|---|
| `task` | `labels` (`five-class` or `binary`), `rows` (largest pool), `balance_classes`, `max_per_class`, `cap_includes_test`, `scaler`, `data_file`, `data_sha256`, `seed` | Which beats form the pool (section 5) |
| `reservoir` | `type` (`bioreservoir` or `esn`) and `params`; or `study`, `trial`, `archive` for the parameters an Optuna trial ran with; or `states_from` to reuse the saved states of another run | `seed` sets the kernel, the input weights and the noise |
| `features` | `aggregation` (`mean`, `final`, `last_N:K`, `first_N:K`), `window` | How a trajectory becomes a feature vector |
| `readout` | `classifier` (`Ridge`, `LR`, `SVM`, `KNN`, `RF`, `DT`, `GB`, `MLP`, `Bayes`, `Perceptron`) and `params`; or `study`, `trial`; `scale` (default true); `search` and `inner_folds` | `scale` standardises the features on each training fold; `search` picks hyperparameters by cross-validation inside each training fold |
| `cv` | `folds`, `primary_average` (`macro`), `noise_rate`, `noise_ratio`, `profile`, `seed` | Stratified k-fold over the pool; augmentation of the training folds only |
| `outputs` | `save_states` | Keep the features in the fold files, to refit readouts later |
| `test` | `split` (`validation-1`, `validation-2` or `test`) | Fit on the whole pool and evaluate on a held-out split instead of cross-validating (section 6) |
| `preset` | `original_v0_1` | The `v0.1` code path; explicit values in the config win |
| `reference` | a run | What the run should match (used by the benchmarks) |

The headline configs are in `experiments/benchmarks/v0.1/`. The run below is the `v0.1` model at small size, with a ridge readout on standardised features (`scale: true` overrides the preset, which turns scaling off as `v0.1` did).

In [ ]:
from training import run

base = {
    "name": "tutorial-coupled",
    "description": "The v0.1 model at small size: 50 oscillators, 300 beats, ridge readout.",
    "preset": "original_v0_1",
    "task": {"labels": "five-class", "rows": 300},
    "reservoir": {"type": "bioreservoir", "params": RESERVOIR},
    "features": {"aggregation": "mean"},
    "readout": {"classifier": "Ridge", "params": {"alpha": 1.0}, "scale": True},
    "cv": {"folds": 3, "profile": True},
    "outputs": {"save_states": True},
}
config_file = OUT / "tutorial-coupled.yaml"
config_file.parent.mkdir(parents=True, exist_ok=True)
config_file.write_text(yaml.safe_dump(base, sort_keys=False))
resolved = cli("training.run", "resolve", str(config_file.relative_to(ROOT)), show=80)

The resolved config is what a run records. From a terminal, on committed code:

```bash
python -m training.run run results/tutorial/tutorial-coupled.yaml --processes 3
```

`--processes` runs the folds in parallel worker processes, `--save-states` keeps the features, `--out` names the run folder, and `--allow-dirty` accepts uncommitted code and marks the run as not citable. Each run also appends a line to `registry.jsonl` at the repository root. The notebook calls the same function from Python, with its own registry.

In [ ]:
def run_config(cfg: dict) -> Path:
    """What `python -m training.run run` does, into results/tutorial/runs/<name> and the
    tutorial's registry. Notebook runs are exploratory, so uncommitted code is allowed."""
    return run.execute(cfg, out=fresh(RUNS / cfg["name"]), allow_dirty=True, registry=REGISTRY)


run_a = run_config(base)  # about a minute: 3 folds, each runs 300 beats through 50 oscillators
for path in sorted(run_a.rglob("*")):
    print(path.relative_to(run_a))

A run folder holds:

- `config.yaml`: the resolved config; `python -m training.run run <folder>/config.yaml` repeats the run;
- `metrics.json`: metrics per fold, their mean and sample SD, the pooled metrics, and timings;
- `fold-<k>.npz`: labels, predictions, the positions of the test beats in the pool (`test_index`), and the features when `save_states` is on;
- `figures/`: the confusion matrix over all folds and the per-fold metrics;
- `run.log` and `manifest.json`: code version (commit, dirty flag), data and lock-file checksums, environment, timing and the checksum of every output.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, name in zip(axes, ("confusion_matrix.png", "fold_metrics.png"), strict=True):
    ax.imshow(plt.imread(run_a / "figures" / name))
    ax.axis("off")
plt.show()

## 3. Reading a run: per-fold, per-class and timing metrics

`metrics.json` holds:

- `folds`: for each fold, the number of test beats, `f1` (the primary average, macro F1), `f1_macro`, `f1_weighted`, accuracy, precision, recall, MCC, Cohen's kappa, the fold's wall time `runtime_s` and its confusion matrix (and, with `readout.search`, the hyperparameters chosen in that fold);
- `mean` and `sd`: over the folds; every SD is a sample SD (ddof = 1);
- `pooled`: the same metrics on the predictions of all folds together;
- `profiling` and `fold_runtimes`, when `cv.profile` is on.

In [ ]:
metrics = json.loads((run_a / "metrics.json").read_text())
per_fold = pd.DataFrame(metrics["folds"]).set_index("fold")
print(per_fold[["n_test", "f1", "accuracy", "mcc", "kappa", "runtime_s"]].round(3), "\n")
pooled = {key: value for key, value in metrics["pooled"].items() if key != "confusion_matrix"}
summary = pd.DataFrame({"mean": metrics["mean"], "sample SD": metrics["sd"], "pooled": pooled})
print(summary.round(4))

**Per class.** `metrics.json` keeps the confusion matrices; precision, recall and F1 per class come from the labels and predictions in the fold files. `training.compare.load_folds` reads them for any run, including archived ones.

In [ ]:
from sklearn.metrics import classification_report, f1_score

from training.compare import load_folds


def classes(Y) -> np.ndarray:
    """Class indices from one-hot labels or predictions of any shape."""
    Y = np.asarray(Y)
    return np.argmax(Y.reshape(len(Y), -1), axis=1)


folds_a = load_folds(run_a)  # one entry per fold: Y_test, Y_pred and test_index
y_true = np.concatenate([classes(fold.Y_test) for fold in folds_a])
y_pred = np.concatenate([classes(fold.Y_pred) for fold in folds_a])
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

per_class = pd.DataFrame(
    [f1_score(classes(fold.Y_test), classes(fold.Y_pred), labels=list(range(5)), average=None,
              zero_division=0) for fold in folds_a],
    columns=CLASS_NAMES)
print("F1 per class over the folds (std is the sample SD):")
print(per_class.agg(["mean", "std"]).round(3))

The same tables, plus plots under `results/figures/metrics/<run>/` (here `results/tutorial/figures/...`), come from a script that works on any set of fold files:

In [ ]:
analysis = cli("scripts/analysis/analyse_fold_results.py", "--glob",
               f"{run_a.relative_to(ROOT)}/fold-*.npz", "--plots", "confusion,class", show=30)

**Timing.**

- `runtime_s` of a fold is its whole wall time: the reservoir on the training beats, the readout fit, the reservoir on the test beats and the prediction.
- With `cv.profile: true`, `profiling` splits it into `train_reservoir`, `fit_readout`, `val_reservoir` and `predict`, and per beat into `instance/reservoir_run` and `instance/state_aggregation` (mean per fold, sample SD over folds, calls per fold); `fold_runtimes` summarises the fold times.
- `manifest.json` has the wall time of the whole run, the peak RAM and the machine (CPU model and core counts; never host names).
- Inside one reservoir, `BioReservoir(diagnostics=True)` and `get_timing_summary()` split each input step into the input computation and the solver.
- For Optuna trials: `python -m optimization.optimize evaluate ... --show timings,resources`.

In [ ]:
profile = pd.DataFrame(metrics["profiling"]).T
print(profile[["mean_s", "std_s", "calls_per_fold"]].round(3), "\n")
per_beat = metrics["profiling"]["instance/reservoir_run"]
print(f"reservoir time per beat: {1000 * per_beat['total_s'] / per_beat['calls_total']:.0f} ms")
manifest = json.loads((run_a / "manifest.json").read_text())
environment = manifest["environment"]
print(f"whole run {manifest['timing']['wall_s']} s, peak RAM {manifest['peak_ram_gb']} GB, "
      f"{environment['cpu_model']} ({environment['cpu_count_physical']} cores)")

## 4. Comparing runs

`python -m training.run compare A B [--out DIR]` (or `training.compare.compare`) recomputes the metrics of both runs from their fold files, so runs from any code version compare on the same basis. A run can be a folder, a registry run id, a folder name under `results/runs`, or `archive:<version>/<run>`.

- **Same beats in the same folds** (same task, seeds and CV settings): the predictions are paired beat by beat. You get the per-fold differences, the share of beats on which both runs agree, and a paired bootstrap 95% CI of the pooled macro-F1 difference (B − A). This is the case for another readout or another reservoir on the same task.
- **Different beats** (another pool, task or seed): only the per-fold metrics and their means and SDs are shown. Compare such runs by mean ± SD, or better, set them up to share the beats.

### 4.1 Another readout on the same states

`reservoir: {states_from: <run>}` refits only the readout, on the states another run saved (`outputs.save_states`). It takes seconds instead of re-running the reservoir, and the folds are the same by construction.

In [ ]:
from training.compare import compare, format_comparison

forest = {
    "name": "tutorial-coupled-rf",
    "description": "A random forest on the saved states of tutorial-coupled.",
    "reservoir": {"states_from": str(run_a)},
    "readout": {"classifier": "RF",
                "params": {"n_estimators": 300, "class_weight": "balanced", "random_state": SEED}},
    "cv": {"folds": 3},
}
run_b = run_config(forest)
print(format_comparison(compare(run_a, run_b, out=fresh(OUT / "compare" / "ridge-vs-forest"))))

### 4.2 Another reservoir on the same beats

The uncoupled array (`rc_scaling: 0`) and an echo state network of the same size see exactly the same beats in the same folds, so these comparisons are paired too. Below, B − A is coupled minus uncoupled. `compare --out` also writes paired plots (per-fold F1, both confusion matrices and, when both runs saved states, the features of a few beats).

In [ ]:
uncoupled = {**base, "name": "tutorial-uncoupled",
             "description": "As tutorial-coupled, with the coupling switched off.",
             "reservoir": {"type": "bioreservoir", "params": {**RESERVOIR, "rc_scaling": 0.0}}}
esn = {**base, "name": "tutorial-esn", "description": "A 50-unit echo state network.",
       "reservoir": {"type": "esn", "params": {
           "units": 50, "esn_leak_rate": 0.316, "esn_spectral_radius": 1.70,
           "esn_activation": "tanh", "input_scaling": 0.327, "input_connectivity": 0.625,
           "input_bias": True, "bias_scaling": 0.963, "noise_in": 0.0055, "noise_rc": 0.00094,
           "rc_connectivity": 0.179}}}
run_c = run_config(uncoupled)  # about a minute
run_d = run_config(esn)  # seconds
print(format_comparison(compare(run_c, run_a, out=fresh(OUT / "compare" / "uncoupled-vs-coupled"))))

scores = {}
for path in (run_c, run_a, run_b, run_d):
    result = json.loads((path / "metrics.json").read_text())
    scores[path.name] = {"macro F1": result["mean"]["f1"], "sample SD": result["sd"]["f1"],
                         "accuracy": result["mean"]["accuracy"]}
print("\n", pd.DataFrame(scores).T.round(3))

### 4.3 Runs on other beats, and the `v0.1` archive (*needs the archive*)

The archived headline runs share the balanced pool and its folds, so the archived BioReservoir and ESN runs compare beat by beat (paired by their test labels, because the `v0.1` fold files did not store beat positions). The small tutorial run uses other beats and other folds, so it does not pair with them.

In [ ]:
if (settings.runs_dir("v0.1") / "P4-readout-balanced-best").is_dir():
    print(format_comparison(compare("archive:v0.1/ESN-readout-best",
                                    "archive:v0.1/P4-readout-balanced-best")), "\n")
    print(format_comparison(compare("archive:v0.1/P4-readout-balanced-best", run_a)))
else:
    print("The v0.1 archive is not available here (BIORESERVOIR_ARCHIVE_DIR): skipped.")

### 4.4 The run registry

Every run appends a line to a registry: here `results/tutorial/registry.jsonl`, for real runs `registry.jsonl` at the repository root. For real runs:

```bash
python -m utils.provenance list [--status ok] [--used]
python -m utils.provenance show RUN_ID
python -m utils.provenance diff RUN_A RUN_B          # metric differences and the manifest fields that differ
python -m utils.provenance mark RUN_ID ok --uses "Table 2, balanced"
python -m utils.provenance mark RUN_ID superseded --by NEW_RUN_ID --note "re-run with the new coupling"
```

Runs are never deleted; status changes are appended. The same functions work on the tutorial's registry:

In [ ]:
from utils import provenance

registry = provenance.load_registry(REGISTRY)
for run_id, entry in registry.items():
    print(run_id, entry["status"], "" if entry["citable"] else "(not citable)", entry["metrics"])

manifests = tuple(json.loads((path / "manifest.json").read_text()) for path in (run_a, run_c))
difference = provenance.diff_runs(registry[manifests[0]["run_id"]],
                                  registry[manifests[1]["run_id"]], manifests)
print("\nmetric differences, uncoupled - coupled:",
      {key: round(value, 4) for key, value in difference["metrics"].items()})
print("config fields that differ:", [key for key in difference["manifest"] if key.startswith("spec.")])

## 5. Data and preprocessing

**Source.** The Kaggle version of the MIT-BIH Arrhythmia Database (Kachuee et al. 2018): 109,446 single beats of 187 samples at 125 Hz (about 1.5 s), each scaled to [0, 1] and zero-padded at the end, labelled N (normal), S (supraventricular), V (ventricular), F (fusion) or Q (unknown). Beats of the same patients appear in every fold, so the evaluation is intra-patient.

**Preparation.** `python -m utils.preprocessing` finds the two CSV files (in `data/ecg/`, in `KAGGLE_HEARTBEAT_DIR`, or by a Kaggle download with the credentials in `config.env`), joins them and writes two caches: `ecg_data.npz` (labels 0-4) and `binary_ecg_data.npz` (0 normal, 1 any arrhythmia). It prints a content checksum, which `tests/integration/test_dataset.py` checks.

**Building a pool** (`load_ecg_data`, called by `run.load_task`):

1. A test part: from each class, 20% of the smallest class (160 beats per class for five classes).
2. A training part: the rest, balanced by undersampling to the smallest class (`balance_classes`) or capped per class (`max_per_class`), then cut to `rows` beats.
3. Labels one-hot encoded, optional noise augmentation, and scaling (`scaler`).
4. The run framework joins the two parts into one pool (at most `rows` beats) and runs stratified k-fold cross-validation on it. Scaling is repeated per fold on the fold's training beats (the default per-beat scaler does not depend on other beats), and augmentation, if any, is added to the training folds only.

`rows` is an upper bound, so the loader's warning that the requested rows "were capped" is expected.

| Option | Meaning | Default |
|---|---|---|
| `scaler` | `sequence_zscore` (each beat to mean 0 and SD 1), `sequence_minmax` (each beat to [0, 1]), `sequence_minmax_pos` (each beat to [0.1, 0.9]), `minmax` and `standard` (per time step, fitted on training beats), `none` | `sequence_zscore` |
| `balance_classes` | Undersample every class to the smallest | true |
| `max_per_class` | Cap per class; smaller classes keep all their beats | none |
| `cap_includes_test` | The cap counts the test beats too; `false` is the `v0.1` rule | true |
| `rows` | Largest pool | 4,015 |
| `labels` | `five-class` or `binary` | `five-class` |
| `data_file`, `data_sha256` | Read another dataset file (relative to `data/`), checked by its SHA-256 | none |
| `cv.noise_rate`, `cv.noise_ratio` | Per class, add `noise_ratio` × (class size) noisy copies of training beats, with Gaussian noise of `noise_rate` × the beat's SD | 0, 0 |

In [ ]:
from utils.preprocessing import dataset_sha256

dataset = settings.data_dir() / "ecg" / "ecg_data.npz"
with np.load(dataset, allow_pickle=True) as data:
    X_all, y_all = np.asarray(data["X"]), np.asarray(data["Y"]).astype(int).ravel()
print(X_all.shape, dict(zip(CLASS_NAMES, np.bincount(y_all).tolist(), strict=True)))
print("content SHA-256:", dataset_sha256(dataset))

The pools of the headline tasks, built as a run builds them. Five-class counts are N, S, V, F, Q; binary counts are normal, arrhythmia. Both binary pools have 37,714 beats, but the `v0.1` runs read a pre-balanced file written by an earlier loader, with other normal beats; the binary benchmark config names that file.

In [ ]:
def pool(**task) -> np.ndarray:
    """The class of every beat in a task's pool, as a run builds it."""
    _, Y, _ = run.load_task({**run.DEFAULTS["task"], "seed": SEED, **task})
    return classes(Y)


tasks = {
    "balanced (v0.1 headline)": {"rows": 4015},
    "unbalanced 5x, cap counts test beats": {"rows": 50000, "max_per_class": 4015},
    "unbalanced 5x, v0.1 cap rule": {"rows": 50000, "max_per_class": 4015,
                                     "cap_includes_test": False},
    "binary, prepared dataset": {"labels": "binary", "rows": 50000},
}
if (settings.data_dir() / "ecg_archive" / "binary_ecg_data.npz").is_file():
    tasks["binary, v0.1 file"] = {"labels": "binary", "rows": 50000,
                                  "data_file": "ecg_archive/binary_ecg_data.npz"}
pools = {name: pool(**task) for name, task in tasks.items()}
for name, labels in pools.items():
    print(f"{name:38s} {len(labels):6d} beats, per class {np.bincount(labels).tolist()}")

In [ ]:
from utils.preprocessing import standardize_data

raw_beat = X_all[np.flatnonzero(y_all == 2)[0]][None, :, None]  # a V beat as stored
fig, ax = plt.subplots(figsize=(9, 3.5))
for scaler in ("none", "sequence_minmax_pos", "sequence_zscore"):
    scaled, _ = standardize_data(raw_beat, raw_beat, scaler_type=scaler)
    ax.plot(scaled[0, :, 0], label=scaler)
ax.set(xlabel="sample (125 Hz)", title="One V beat under three per-beat scalers")
ax.legend()
plt.show()

## 6. Held-out beats

**Why they are needed.** Every `v0.1` optimisation study drew its beats from one balanced pool of 4,015 beats, 803 per class, and every hyperparameter and readout was chosen by cross-validation on that pool. A cross-validated score on the same pool is therefore a little optimistic: it is the score of the best of many choices. Beats outside the pool were never used for any choice, so a score on them is an honest estimate of how the chosen design does on new beats (of the same patients).

**What is left.** The pool holds all 803 F beats of the dataset, so the held-out splits have only N, S, V and Q beats, and are scored with macro-F1 over those four classes. Of those, S is the limiting class: 2,779 S beats in all, 803 of them in the pool.

**The three splits** (`utils/heldout.py`; beat IDs and checksums in `data/splits/heldout.json`, tracked in git). Each is a list of beat IDs (row numbers in `ecg_data.npz`) drawn at random, balanced over N, S, V and Q, from the beats outside the pool and outside the earlier splits:

- **`validation-1` and `validation-2`** (draws 1 and 2) take 803 beats per class, the per-class size of the pool. They have been evaluated during model development, to check that gains measured by cross-validation hold on new beats. Because they have informed choices, they are validation sets, not test sets.
- **`test`** (draw 3) takes what the limiting class has left: 2,779 − 3 × 803 = 370 S beats, so 370 per class, 1,480 beats. It is smaller than the validation splits only because S runs out. It was recorded before any evaluation and is evaluated once, at the end.

**How a run is evaluated on a split.** A run config with `test: {split: <name>}` does not cross-validate: it fits the readout on the whole pool, runs the reservoir on the split's beats, and scores them with macro-F1 over the split's classes (an F prediction counts as an error) and a bootstrap 95% CI. The run refuses split beats that are also in its pool, records the split's beat IDs in its manifest, and pairs beat by beat with other runs on the same split in `compare`. The `test` split also needs `--final-test`.

**The final evaluation:**

1. Choose everything by cross-validation on the pool: the model, its parameters, the readout and its regularisation (nested cross-validation).
2. Check on a validation split that a choice generalises, if needed; never tune on it.
3. Freeze the design and commit the configs.
4. Run each final config once with `test: {split: test}` and `--final-test`, and compare the models on the same test beats with `compare` (paired bootstrap CIs).
5. Report the result whatever it is.

Never tune anything on a held-out split, and never change `heldout.json`.

In [ ]:
from utils import heldout

record = json.loads(heldout.SPLITS_FILE.read_text())
usage = pd.DataFrame({"all beats": np.bincount(y_all),
                      "CV pool": np.bincount(pools["balanced (v0.1 headline)"], minlength=5)},
                     index=CLASS_NAMES)
for draw in record["draws"].values():
    usage[draw["split"]] = [draw["counts"].get(name, 0) for name in CLASS_NAMES]
usage["never used"] = usage["all beats"] - usage.drop(columns="all beats").sum(axis=1)
print(usage, "\n")
for draw in record["draws"].values():
    print(f"{draw['split']:13s} seed {draw['seed']}, {len(draw['beat_ids'])} beats: "
          f"{draw['status']}")

In [ ]:
heldout_check = cli("utils.heldout", "--check")  # rebuilds the draws from the data, checks their checksums

A held-out evaluation is the same config with a `test` section. The cell below shows how it resolves; set `RUN_VALIDATION = True` to run it on `validation-1` (3,212 beats through 50 oscillators: about four minutes). From a terminal: `python -m training.run run <config>`, plus `--final-test` for the `test` split.

In [ ]:
RUN_VALIDATION = False

validation = {**base, "name": "tutorial-validation-1", "test": {"split": "validation-1"}}
print("test section:", run.resolve(validation)["test"])
if RUN_VALIDATION:
    run_v = run_config(validation)
    evaluated = json.loads((run_v / "metrics.json").read_text())
    print("macro F1 over N, S, V, Q:", round(evaluated["mean"]["f1"], 3),
          "| 95% CI", [round(value, 3) for value in evaluated["test"]["f1_macro_ci95"]])

## 7. The optimisation framework

**Study registry** (`optimization/studies.py`). `BASELINE` holds a value for every parameter. Each entry of `STUDY_REGISTRY` names the parameters to search (`optimize`, as Optuna suggest specs), the ones to fix (`fixed`), the sampler, the trial budget, the folds, the pool size and the labels. A trial's parameters are `BASELINE`, then `fixed`, then the suggested values.

**Study database.** `results/optuna/optuna-<study>.db` (SQLite). Every trial stores all its parameters, its metrics per fold and per class, timings and the environment as user attributes, so the database alone rebuilds a trial: `reservoir: {study: <study>, trial: <number>}` in a run config. The frozen `v0.1` databases are read with `--archive v0.1`.

**Three kinds of study:**

- `--type training`: the reservoir's parameters, scored by cross-validation with a fixed KNN readout on unscaled features (the `v0.1` protocol, in `optimization/training.py`). A study that scores another readout or adds measurement noise needs its own objective.
- `--type readout`: the readout's family and hyperparameters, on the saved states of one run (no reservoir simulation).
- `--type baseline`: classifiers on the raw beats, without a reservoir.

**The `v0.1` workflow.** Phase 1 studies of one group of parameters each, then `BASELINE` updated, then the joint Phase 2 study (`P2-distance`), then `rerun-best` to save the best trial's states, then the Phase 4 readout studies on those states, then `apply-best`, which gives the headline runs.

```bash
python -m optimization.optimize list-studies
python -m optimization.optimize research --type training --study P2-distance --processes 32
#   --trials N --instances N --folds N --binary --sampler tpe|random|grid --fold_workers N
#   --save_states --study_name NAME (another database name for the same registry entry)
python -m optimization.optimize evaluate --type training --study P2-distance --show params,metrics,timings --plots slice,importance
python -m optimization.optimize evaluate --type training --study P2-distance --archive v0.1
python -m optimization.optimize rerun-best --study P2-distance --folds 5 --processes 5
python -m optimization.optimize research --type readout --study_name P4-readout-balanced --trial_name P2-distance-best --classifiers KNN RF Ridge LR --trials 500 --processes 32
python -m optimization.optimize apply-best --study_name P4-readout-balanced --trial_name P2-distance-best
python -m optimization.optimize research --type baseline --study_name B1-balanced --trials 500 --processes 32
```

`--processes N` starts N workers that share one database and share out the trials. `rerun-best` and `apply-best` write run folders (`results/runs/<study>-best`) through the run framework. On remote hosts, `python -m optimization.deploy --host host1 --study P2-distance` starts the same command in a detached session and `python -m optimization.collect --host host1 --study P2-distance` copies the results back (README, "Remote Deployment").

**A study for a new model.** Add an entry to `STUDY_REGISTRY` with the new parameters in `optimize` (and anything else the builder needs in `fixed`), make sure `optimization/reservoirs.py` passes the new parameters to the node, and run `research` on it. Because trials record every value they ran with, `reservoir: {study: <name>}` in a run config rebuilds the best trial.

In [ ]:
from optimization.studies import get_study

studies = cli("optimization.optimize", "list-studies", show=40)
print(json.dumps(get_study("P1-coupling"), indent=1))

**A small reservoir study.** The registry's `ESN-balanced` entry is used here, on 100 beats and two folds, because its trials take about a minute each at this size (the scores mean nothing at this size); a BioReservoir study works the same way, but each trial takes minutes at 250 oscillators and hours at 1,000. This notebook points `BIORESERVOIR_RESULTS_DIR` at `results/tutorial`, so the database is `results/tutorial/optuna/optuna-ESN-balanced.db`. Running `research` again adds trials to the same study.

In [ ]:
(OUT / "optuna" / "optuna-ESN-balanced.db").unlink(missing_ok=True)
study_log = cli("optimization.optimize", "research", "--type", "training", "--study",
                "ESN-balanced", "--trials", "3", "--instances", "100", "--folds", "2", show=3)
study_report = cli("optimization.optimize", "evaluate", "--type", "training", "--study",
                   "ESN-balanced", "--top_n", "3", "--show", "params,metrics", "--plots", "",
                   show=30)

A run config can take its reservoir from the best trial. `resolve` reads the trial and fills in the task, features and augmentation it ran with:

In [ ]:
from_study = {
    "name": "tutorial-esn-best",
    "description": "The best trial of the tutorial's ESN-balanced study, with a ridge readout.",
    "reservoir": {"study": "ESN-balanced"},
    "readout": {"classifier": "Ridge", "params": {"alpha": 1.0}},
}
print(run.dump_config(run.resolve(from_study)))

**A readout study** on the saved states of `tutorial-coupled`: `--trial_name` names the run folder under `results/runs` (`results/tutorial/runs` here). Its trials only fit readouts, so they take seconds. The readout objective fits on unscaled features, as in `v0.1`, whereas the run framework scales them by default. `apply-best` would turn the best trial into a run; it writes to the repository's registry, so it is not called here.

In [ ]:
(OUT / "optuna" / "optuna-tutorial-readout.db").unlink(missing_ok=True)
readout_log = cli("optimization.optimize", "research", "--type", "readout", "--study_name",
                  "tutorial-readout", "--trial_name", "tutorial-coupled", "--classifiers",
                  "Ridge", "KNN", "RF", "--trials", "8", show=3)
readout_report = cli("optimization.optimize", "evaluate", "--type", "readout", "--study_name",
                     "tutorial-readout", "--top_n", "3", "--plots", "", show=25)

**An archived study** (*needs the archive*): the best trials of the `v0.1` Phase 2 study, read-only.

In [ ]:
if settings.optuna_db("P2-distance", "v0.1").is_file():
    archived_report = cli("optimization.optimize", "evaluate", "--type", "training", "--study",
                          "P2-distance", "--archive", "v0.1", "--top_n", "3", "--plots", "",
                          show=20)
else:
    print("The v0.1 archive is not available here (BIORESERVOIR_ARCHIVE_DIR): skipped.")

## 8. Checking behaviour against `v0.1`, and changing it on purpose

**Code defaults and presets.** The code defaults are the corrected behaviour (CHANGELOG, "Fixed"). `preset: original_v0_1` in a run config, and `--preset original_v0_1` in the ablation and heterogeneity scripts, restore the `v0.1` code path. The four headline configs in `experiments/benchmarks/v0.1/` use it, and each names the archived run it must reproduce.

| Check | What it compares | Command | When |
|---|---|---|---|
| Unit tests | Each module, including a test for every fixed bug | `pytest tests/unit` | Every change (CI) |
| Model regression test | Reservoir states and features for five synthetic beats (N = 20, `v0.1` settings) against outputs of the `v0.1` code, to 1e-6 | part of `pytest tests/unit` | Every change (CI) |
| Pipeline regression test | Two-fold CV, N = 200, 600 real beats: macro-F1 within 0.02 of `v0.1` | `pytest tests/integration` | Every change (CI) |
| Benchmark configs | The headline configs build the archived models and rebuild the archived folds | `pytest tests/integration` (needs the archive) | Changes to data, models or the runner |
| Benchmarks | The four headline runs at full size, against the archive fold by fold: agreement ≥ 0.99, macro-F1 difference ≤ 0.005 | `BIORESERVOIR_BENCHMARK_PROCESSES=5 pytest -m benchmark` | On a server, after changes to the solver, node, data or runner, and before a release |
| Run comparison | Any two runs | `python -m training.run compare A B` | Whenever you want to know how much something changed |

A change meant to keep behaviour (a refactor, a speed-up, a dependency pin) must pass all of these unchanged; a benchmark rerun against its archived run shows agreement 1.0000 and a difference of 0.

In [ ]:
print(json.dumps(run.PRESETS, indent=1))
for path in sorted((ROOT / "experiments" / "benchmarks" / "v0.1").glob("*.yaml")):
    benchmark = run.load_config(path)
    print(f"{path.name:18s} preset {benchmark.get('preset')}, reference {benchmark.get('reference')}")
regression = cli("pytest", "tests/unit/test_model_regression.py", "-q", "-p", "no:cacheprovider",
                 show=6)

**Changing behaviour on purpose, for example the coupling.** The aim is to show that nothing else moved, and to measure what did:

1. **Add rather than replace.** Put the new behaviour behind an option whose default keeps the current path (for example a `coupling` parameter on `BioReservoir`, with the abstract coupling as one of its values). Every existing test, the model regression test and the benchmark configs must then pass without changes: that is the evidence that the old path is untouched.
2. **Test the new physics directly**, through the properties the model must have rather than its outputs. For AHL diffusion between colonies: the exchange terms sum to zero when the conductances are symmetric, vanish when all colonies have the same He, touch no variable other than He, and keep every concentration non-negative for non-negative input. Add an equivalence test against an independent implementation on a small case (for example a few colonies integrated with SciPy).
3. **Give it a name.** Add a preset to `PRESETS` in `training/run.py` (for example `qs_diffusion`), so that a config says which model it uses. Add the preset to `tests/regression_reference.py` with its own reference file, generated from a tagged commit once the model is frozen; leave the `v0.1` reference alone.
4. **Check the regime before the scores** with the diagnostics of section 1 or `scripts/analysis/drive_bounds.py <config>`: drive range, lowest and highest states, drive shares.
5. **Measure the change on the same beats.** Write configs that differ only in the reservoir (same task, seeds and folds), run them and `compare` them: per-fold differences, agreement and a paired bootstrap CI. The cell below does this for the corrected code against the `v0.1` code path.
6. **Re-tune on the CV pool only**, if needed (a study, section 7), then check on a validation split and freeze the design.
7. **Freeze a new reference.** Write benchmark configs for the frozen design (for example `experiments/benchmarks/v0.2/`) whose `reference` is the frozen runs; `pytest -m benchmark` then guards them as it guards `v0.1`.
8. **Record it**: a `[science]` CHANGELOG entry with the measured effect, `provenance mark ... --uses` for the runs behind the paper's tables and figures, and `mark ... superseded --by` for the runs they replace.

The `v0.1` path stays reproducible throughout, through the `original_v0_1` preset, its regression reference and its benchmark configs.

In [ ]:
corrected = {key: value for key, value in base.items() if key != "preset"}
corrected.update(name="tutorial-corrected",
                 description="As tutorial-coupled, with the corrected code (no preset).")
run_e = run_config(corrected)  # about a minute
print(format_comparison(compare(run_a, run_e, out=fresh(OUT / "compare" / "v0.1-vs-corrected"))))

The agreement is the share of beats that keep their predicted class; the CI says whether macro-F1 moved by more than chance on these beats. At this size the CI is wide. At full size (4,015 beats, 1,000 oscillators) the corrected code with the `v0.1` gains scores 0.866 instead of 0.892 macro-F1 (CHANGELOG), and section 1 shows why the gains need re-tuning.

## 9. Figures and the scripts that make them

Every run folder has its own confusion matrix and per-fold plot, and `compare --out` writes paired plots. The paper's figures come from these scripts:

| Figure | Made by | Depends on the coupling model? |
|---|---|---|
| Optuna slice plots of the `v0.1` studies (Phases 1, 2 and 4, ESN) | `python -m optimization.optimize evaluate --type training --study <study> --archive v0.1 --plots slice` (`--type readout` for Phase 4) | No: they document the `v0.1` studies |
| Headline confusion matrices | each run's `figures/confusion_matrix.png`; `scripts/analysis/analyse_fold_results.py --glob '<run>/fold-*.npz' --plots confusion` | Yes. No script assembles the multi-panel figure |
| Per-class tables, MCC and kappa | `scripts/compute_headline_metrics.py` | Yes. It hard-codes the `v0.1` runs and readouts |
| Model overview: oscillator response at several input scales | `scripts/figures/visualise_dde.py` (the schematic panels are drawn by hand) | Yes, if the input route changes |
| Drive, states and topology | `scripts/figures/visualise_reservoir_states.py --plot-drive --plot-topology`, `scripts/figures/visualise_topology.py`, `scripts/analysis/drive_bounds.py` | Yes. They assume the abstract coupling (input term plus a signed kernel applied to the states) |
| Dataset, augmentation and scalers | `scripts/figures/visualise_dataset.py` | No |
| Solver convergence (SI) | `scripts/convergence_analysis.py` | Yes. It analyses the frozen-delay solver of `v0.1` |
| Heterogeneity (SI) | `scripts/run_heterogeneity.py`, then `utils.visualisation.plot_heterogeneity_robustness` | Yes. The plot reads an Optuna study (`P3-heterogeneity`), not the script's runs, and no command makes it |
| Ablations (table) | `scripts/run_ablation.py` | Yes. It hard-codes the `v0.1` reservoir |
| Readout tables S13-S15 | `scripts/analysis/readout_tables.py` | No: the `v0.1` Phase 4 studies |

A new coupling model therefore needs new runs for most figures, updates to the drive, topology and convergence scripts, and new scripts for figures that combine several runs. Figure scripts that read run folders or run configs, rather than hard-coded parameters, turn the next model change into a change of configs.

## Command summary

```bash
uv sync --extra dev && python -m utils.preprocessing             # environment and data
python -m training.run resolve CONFIG.yaml                       # the explicit config
python -m training.run run CONFIG.yaml --processes 5 [--save-states] [--allow-dirty]
python -m training.run run FINAL.yaml --final-test               # a config with test: {split: test}
python scripts/analysis/analyse_fold_results.py --glob 'results/runs/RUN/fold-*.npz' --plots
python -m training.run compare RUN_A RUN_B --out results/compare/NAME
python -m utils.provenance list | show RUN | diff RUN_A RUN_B | mark RUN STATUS
python -m utils.heldout --check                                  # held-out splits
python -m optimization.optimize list-studies | research | evaluate | rerun-best | apply-best
python scripts/analysis/drive_bounds.py CONFIG.yaml              # drive and state ranges
pytest tests/unit && pytest tests/integration                    # every change
BIORESERVOIR_BENCHMARK_PROCESSES=5 pytest -m benchmark           # headline reruns, on a server
```

To remove everything this notebook wrote, run the cell below.

In [ ]:
# shutil.rmtree(OUT)  # uncomment to delete results/tutorial/